In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, FloatType, LongType

spark = SparkSession.builder\
        .appName("e-commerce_analysis")\
        .master("local[*]")\
        .config("spark.executor.memory", "2g")\
        .config("spark.driver.memory", "1g")\
        .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/10/30 18:06:21 WARN Utils: Your hostname, Dune, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/10/30 18:06:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/30 18:06:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
data_path = "/home/lpascual/Projects/E-Commerce_DWH/data/bronze/landing/2019-Oct-enriched.csv"
output_dir = "/home/lpascual/Projects/E-Commerce_DWH/data/bronze/parquet/"

In [3]:
schema = StructType([
    StructField("timestamp", DateType()),
    StructField("event_type", StringType()),
    StructField("product_id", IntegerType()),
    StructField("category_id", LongType()),
    StructField("category_code", StringType()),
    StructField("brand", StringType()),
    StructField("price", FloatType()),
    StructField("user_id", LongType()),
    StructField("user_session", StringType()),
    StructField("name", StringType()),
    StructField("username", StringType()),
    StructField("email", StringType()),
    StructField("address", StringType()),
    StructField("city", StringType()),
    StructField("country", StringType()),
    StructField("state_code", StringType()),
    StructField("zip_code", StringType()),
    StructField("sex", StringType()),
    StructField("product_name", StringType()),
    StructField("product_description", StringType())
])

In [4]:
df = spark.read.format("csv")\
    .schema(schema)\
    .option("header", "true")\
    .load(data_path)\
    .select("timestamp", "event_type", "product_id", "price", "user_id", "user_session")
print("Number of partitions:", df.rdd.getNumPartitions())

Number of partitions: 105


In [5]:
# Coalesce into fewer partitions
df.write.partitionBy("timestamp").parquet(path=output_dir, mode="overwrite")

25/10/30 18:06:31 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: event_time, event_type, product_id, price, user_id, user_session
 Schema: timestamp, event_type, product_id, price, user_id, user_session
Expected: timestamp but found: event_time
CSV file: file:///home/lpascual/Projects/E-Commerce_DWH/data/bronze/landing/2019-Oct-enriched.csv
25/10/30 18:06:50 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
25/10/30 18:06:50 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
25/10/30 18:06:50 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
25/10/30 18:06:50 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
25/10/30

In [6]:
spark.stop()